In [7]:
import numpy as np
import tkinter as tk
from tkinter import messagebox

EMPTY = 0
PLAYER = 1
AI = 2
BOARD_SIZE = 6

# Create a Pente board
def create_board():
    return np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

In [8]:
# Check if the game is over
def is_game_over(board):
    return check_winner(board) != 0 or not np.any(board == EMPTY)

# Check for a winner
def check_winner(board):
    # Check rows, columns, and diagonals for 5 in a row
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if board[row][col] != EMPTY:
                if check_direction(board, row, col, 1, 0) or \
                   check_direction(board, row, col, 0, 1) or \
                   check_direction(board, row, col, 1, 1) or \
                   check_direction(board, row, col, -1, 0) or \
                   check_direction(board, row, col,  0, -1) or \
                   check_direction(board, row, col, -1, -1) or \
                   check_direction(board, row, col, 1, -1):
                    return board[row][col]
    return 0

# Check a specific direction for 5 in a row
def check_direction(board, row, col, d_row, d_col):
    count = 0
    player = board[row][col]
    for i in range(5):
        r, c = row + i * d_row, col + i * d_col
        if 0 <= r < BOARD_SIZE and 0 <= c < BOARD_SIZE and board[r][c] == player:
            count += 1
        else:
            break
    return count == 5

In [9]:
# Get all possible moves
def get_possible_moves(board):
    moves = []
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if board[row][col] == EMPTY:
                moves.append((row, col))
    return moves

In [10]:
# Using Heuristic Search To Calculate The Score
def evaluate_board(board):
    ai_score = evaluate_player_positions(board, AI)
    player_score = evaluate_player_positions(board, PLAYER)

    # AI's advantage should be maximized and PLAYER's minimized
    return ai_score - player_score

# Evaluate positions for a specific player
def evaluate_player_positions(board, player):
    score = 0

    # Check rows, columns, and diagonals for patterns
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if board[row][col] == player:
                # Evaluate horizontal patterns (down)
                score += score_pattern(board, row, col, 1, 0, player)
                # Evaluate vertical patterns (up)
                score += score_pattern(board, row, col, 0, 1, player)
                # Evaluate diagonal patterns (down-right)
                score += score_pattern(board, row, col, 1, 1, player)
                # Evaluate diagonal patterns (down-left)
                score += score_pattern(board, row, col, 1, -1, player)

    return score

# Assign scores to patterns based on the number of stones in a row and openness
def score_pattern(board, row, col, d_row, d_col, player):
    count = 0
    open_ends = 0
    r, c = row, col

    # Count consecutive stones of the same player
    for _ in range(5):
        if 0 <= r < BOARD_SIZE and 0 <= c < BOARD_SIZE and board[r][c] == player:
            count += 1
            r += d_row
            c += d_col
        else:
            break

    # Check for open ends
    if 0 <= r < BOARD_SIZE and 0 <= c < BOARD_SIZE and board[r][c] == EMPTY:
        open_ends += 1

    # Check the opposite direction for open ends
    r, c = row - d_row, col - d_col
    if 0 <= r < BOARD_SIZE and 0 <= c < BOARD_SIZE and board[r][c] == EMPTY:
        open_ends += 1

    # Assign scores based on pattern strength
    if count == 5:
        return 10000  # Winning move
    elif count == 4 and open_ends == 2:
        return 5000  # Open four (guaranteed win next turn)
    elif count == 4 and open_ends == 1:
        return 1000  # Closed four
    elif count == 3 and open_ends == 2:
        return 505  # Open three
    elif count == 3 and open_ends == 1:
        return 100  # Closed three
    elif count == 2 and open_ends == 2:
        return 50  # Open two
    elif count == 2 and open_ends == 1:
        return 20  # Closed two
    return 0  # Other patterns

In [11]:
# Minimax algorithm with alpha-beta pruning
def minimax(board, depth, alpha, beta, is_maximizing):
    if depth == 0 or is_game_over(board):
        return evaluate_board(board)

    if is_maximizing:
        max_eval = float('-inf')
        for move in get_possible_moves(board):
            board[move[0]][move[1]] = AI
            eval = minimax(board, depth - 1, alpha, beta, False)
            board[move[0]][move[1]] = EMPTY
            max_eval = max(max_eval, eval)
            alpha = max(alpha, eval)
            if beta <= alpha:
                break
        return max_eval
    else:
        min_eval = float('inf')
        for move in get_possible_moves(board):
            board[move[0]][move[1]] = PLAYER
            eval = minimax(board, depth - 1, alpha, beta, True)
            board[move[0]][move[1]] = EMPTY
            min_eval = min(min_eval, eval)
            beta = min(beta, eval)
            if beta <= alpha:
                break
        return min_eval

In [12]:
# Find the best move for AI
def find_best_move(board):
    best_score = float('-inf')
    best_move = None
    for move in get_possible_moves(board):
        board[move[0]][move[1]] = AI
        score = minimax(board, depth=2, alpha=float('-inf'), beta=float('inf'), is_maximizing=False)
        board[move[0]][move[1]] = EMPTY
        if score > best_score:
            best_score = score
            best_move = move
    return best_move

In [14]:
# GUI Implementation
class PenteGUI:
    def __init__(self, root):
        self.root = root
        self.board = create_board()
        self.buttons = [[None for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]
        self.create_widgets()

    def create_widgets(self):
        for row in range(BOARD_SIZE):
            for col in range(BOARD_SIZE):
                button = tk.Button(self.root, text="", width=4, height=2, command=lambda r=row, c=col: self.player_move(r, c))
                button.grid(row=row, column=col)
                self.buttons[row][col] = button

    def update_gui(self):
        for row in range(BOARD_SIZE):
            for col in range(BOARD_SIZE):
                if self.board[row][col] == PLAYER:
                    self.buttons[row][col].config(text="X", state=tk.DISABLED)
                elif self.board[row][col] == AI:
                    self.buttons[row][col].config(text="O", state=tk.DISABLED)

    def player_move(self, row, col):
        if self.board[row][col] == EMPTY:  # Ensure the cell is vacant
            self.board[row][col] = PLAYER
            self.update_gui()
            captured = self.capture_stones(row, col, PLAYER)
            print(f"Captured stones: {captured}")

            if is_game_over(self.board):
                self.end_game()
                return

            # Switch to AI move
            self.ai_move()

    def ai_move(self):
        ai_move = find_best_move(self.board)
        if ai_move:
            self.board[ai_move[0]][ai_move[1]] = AI
            self.update_gui()
            captured = self.capture_stones(ai_move[0], ai_move[1], AI)
            print(f"AI captured stones: {captured}")

        if is_game_over(self.board):
            self.end_game()



    def capture_stones(self, row, col, player):
        directions = [
            (-1, 0),
            (1, 0),
            (0, -1),
            (0, 1),
            (-1, -1),
            (1, 1),
            (-1, 1),
            (1, -1),
        ]
        opponent = PLAYER if player == AI else AI
        captured = []

        for d_row, d_col in directions:
            r1, c1 = row + d_row, col + d_col
            r2, c2 = row + 2 * d_row, col + 2 * d_col
            r3, c3 = row + 3 * d_row, col + 3 * d_col
            r4, c4 = row + 4 * d_row, col + 4 * d_col

            if (
                self.is_within_bounds(r1, c1)
                and self.is_within_bounds(r2, c2)
                and self.is_within_bounds(r3, c3)
                and self.is_within_bounds(r4, c4)
                and self.board[r1][c1] == opponent
                and self.board[r2][c2] == opponent
                and self.board[r3][c3] == opponent
                and self.board[r4][c4] == opponent
            ):
                r5, c5 = row + 5 * d_row, col + 5 * d_col
                if self.is_within_bounds(r5, c5) and self.board[r5][c5] == player:
                    captured.append((r1, c1))
                    captured.append((r2, c2))
                    captured.append((r3, c3))

            elif (
                self.is_within_bounds(r1, c1)
                and self.is_within_bounds(r2, c2)
                and self.is_within_bounds(r3, c3)
                and self.board[r1][c1] == opponent
                and self.board[r2][c2] == opponent
                and self.board[r3][c3] == opponent
            ):
                r4, c4 = row + 4 * d_row, col + 4 * d_col
                if self.is_within_bounds(r4, c4) and self.board[r4][c4] == player:
                    captured.append((r1, c1))
                    captured.append((r2, c2))
                    captured.append((r3, c3))
            elif (
                self.is_within_bounds(r1, c1)
                and self.is_within_bounds(r2, c2)
                and self.board[r1][c1] == opponent
                and self.board[r2][c2] == opponent
            ):
                r3, c3 = row + 3 * d_row, col + 3 * d_col
                if self.is_within_bounds(r3, c3) and self.board[r3][c3] == player:
                    captured.append((r1, c1))
                    captured.append((r2, c2))

            elif (
                self.is_within_bounds(r1, c1)
                and self.board[r1][c1] == opponent
            ):
                r2, c2 = row + 2 * d_row, col + 2 * d_col
                if self.is_within_bounds(r2, c2) and self.board[r2][c2] == player:
                    captured.append((r1, c1))
                    
        for r, c in captured:
            self.board[r][c] = EMPTY
            self.buttons[r][c].config(text="", state=tk.NORMAL)

        return captured

    def is_within_bounds(self, row, col):
        return 0 <= row < BOARD_SIZE and 0 <= col < BOARD_SIZE

        
    
    def end_game(self):
        winner = check_winner(self.board)
        if winner == PLAYER:
            messagebox.showinfo("Game Over", "Congratulations, you win!")
        elif winner == AI:
            messagebox.showinfo("Game Over", "AI wins! Better luck next time.")
        else:
            messagebox.showinfo("Game Over", "It's a draw!")
        self.root.quit()


# Start the game
if __name__ == "__main__":
    root = tk.Tk()
    root.title("Pente")
    game = PenteGUI(root)
    root.mainloop()

Captured stones: []
AI captured stones: []
Captured stones: [(1, 1)]
AI captured stones: []
Captured stones: []
AI captured stones: []
Captured stones: []
AI captured stones: []
Captured stones: []
AI captured stones: []
Captured stones: []
AI captured stones: []
Captured stones: [(2, 1)]
AI captured stones: []
Captured stones: []
AI captured stones: []
Captured stones: []
AI captured stones: []
Captured stones: []
AI captured stones: []
Captured stones: []
AI captured stones: [(2, 2)]
Captured stones: []
AI captured stones: [(3, 3)]
